# A quick tour of voltage-clamp models

The model is available in different levels of detail, ranging from the "full" model (level 0) to a greatly simplified one (level 5).
Here, we provide a quick tour of the most important models, and provide guidance for setting parameter values.

_This notebook presents, but does not derive or explain the models. For that, please see the Tutorial Notebooks_.

## Level 5 model

This model calculates membrane potential $V_m$ from command potential $V_c$, and observed current $I_\text{obs}$ from ionic current $I$.

<img src="./img/vc-level-5-post.png" style="margin:auto" />

It includes:

- A voltage offset $E^\dagger_\text{off}$ representing the remaining voltage error after zeroing
- A leak current $I_\text{leak}$
- The effect of series resistance $R_s$ on the membrane potential
- Series resistance compensation, $\alpha$, and prediction, $\beta$, both specified as fractions between 0 and 1.

\begin{align}
5.1. && C_m \dot{V}_m = \frac{V_c + E_\text{off}^\dagger - V_m}{(1 - \beta) R_s} - \frac{1 - \alpha}{1 - \beta}(I + I_\text{leak})
\end{align}

\begin{align}
5.2. && I_\text{obs} = I + I_\text{leak}
\end{align}




| Variable        | Units | Meaning                                                 |
|-----------------|-------|---------------------------------------------------------|
| $V_m$           | mV    | Membrane potential                                      |
| $V_c$           | mV    | Command potential, set by the voltage clamp protocol    |
| $I$             | pA    | Ionic current                                           |
| $I_\text{obs}$  | pA    | Observed (measured/output) current                      |
| $I_\text{leak}$ | pA    | Leak current, e.g. $g_\text{leak}(V_m - I_\text{leak})$ |

The variables $I$ and $I_\text{leak}$ should be supplied by an ionic current model and a leak model, respectively.

| Parameter | Units     | Meaning                                             | Typical value   |
|-----------|-----------|-----------------------------------------------------|-----------------|
| $C_m$     | pF        | Cell capacitance, from the amplifier estimate       | 25 pF           |
| $R_s$     | G$\Omega$ | Series resistance, from the amplifier estimate      | 0.005 G$\Omega$ |
| $\alpha$  | -         | Fraction of $R_s$ compensation, read from amplifier | 0.7             |
| $\beta$   | -         | Fraction of $R_s$ prediction, read from amplifier   | 0.7             |
| $E^\dagger_\text{off}$ | mV | Remaining voltage offset after zeroing        | 0 mV            |

The $C_m$ value corresponds to a small cell, e.g. a HEK or CHO cell.

The $R_s$ value corresponds to a 3M$\Omega$ pipette, with an additional 3M$\Omega$ resistance at the seal.

In a simple simulation $E^\dagger_\text{off}$ is chosen by the user.
In inference settings it may be a parameter inferred from the data.

## Level 4 model

<img src="./img/vc-level-4-post.png" style="margin:auto" />

This level adds

- A distinction between estimated and true membrane capacitance and series resistance
- A state-estimator used to approximate $V_m$ in the prediction pathway

\begin{align}
4.1. && C_m\dot{V}_m = \frac{V_c + E_\text{off}^\dagger - V_m - (\alpha - \beta) R_s^* C_m^* \dot{V}_e}{R_s - \alpha R_s^*} - I_\text{leak} - I
\end{align}

\begin{align}
4.2. && \dot{V}_e &= \frac{V_c - V_e}{(1 - \beta) R_s^* C_m^*}   
\end{align}

\begin{align}
4.3. && I_\text{obs} = I + I_\text{leak} + C_m \dot{V}_m - C_m^* \dot{V}_e
\end{align}

New or changed variables and parameters are

| Variable | Units | Meaning                                          |
|----------|-------|--------------------------------------------------|
| $V_e$    | mV    | Estimate of $V_m$ used in the prediction pathway |

And

| Parameter | Units     | Meaning                                          | Typical value   |
|-----------|-----------|--------------------------------------------------|-----------------|
| $C_m$     | pF        | True cell capacitance, unknown                   | 25 pF           |
| $C^*_m$   | pF        | Estimated cell capacitance, read from amplifier  | 25 pF           |
| $R_s$     | G$\Omega$ | True series resistance, unknown                  | 0.005 G$\Omega$ |
| $R^*_s$   | G$\Omega$ | Estimated series resistance, read from amplifier | 0.005 G$\Omega$ |

For a straightforward simulation, $C_m$ and $R_s$ are chosen by the user.
In an inference setting, $C_m$ and $R_s$ may be inferred from the data long with $E^\dagger_\text{off}$.

## Level 3 model

<img src="./img/vc-level-3-post.png" style="margin:auto" />

This level adds

- An explicit measurement resistor $R_f$
- A first-order low-pass filter in the series resistance compensation pathway
- A first-order low-pass filter over the command potential (the "stimulus filter")
- A first-order low-pass filter over the output

\begin{align}
3.1. && C_m\dot{V}_m = \frac{V_r + E_\text{off}^\dagger - V_m}{R_s} - I_\text{leak} - I
\end{align}

\begin{align}
3.2. && V_r = V_s + \alpha \frac{R_s^*}{R_f} V_\text{rc} + \beta R_s^* C_m^* \dot{V}_e
              = \alpha \frac{R_s^*}{R_f} V_\text{rc} + \frac{V_s - \beta V_e}{1 - \beta}
\end{align}

\begin{align}
3.3. && \dot{V}_e &= \frac{V_s - V_e}{(1 - \beta)R_s^*C_m^*}   
\end{align}

\begin{align}
3.4 && I_1 &= \frac{V_r + E_\text{off}^\dagger - V_m}{R_s} - C_m^* \dot{V_e}
\end{align}

\begin{align}
3.5 && \tau_\text{rc} \dot{V}_\text{rc} &= R_f I_1 - V_\text{rc}
\end{align}

\begin{align}
3.6. && \tau_s \dot{V_s} = V_c - V_s
\end{align}

\begin{align}
3.7. && \tau_o \dot{I}_\text{obs} = I_1 - I_\text{obs}
\end{align}

New variables and parameters are

| Variable      | Units | Meaning                                     |
|---------------|-------|---------------------------------------------|
| $V_r$         | mV    | Reference voltage set by the voltage clamp  |
| $V_s$         | mV    | Filtered command potential                  |
| $V_\text{rc}$ | mV    | Filtered voltage used in $R_s$ compensation |
| $I_1$         | pA    | Unfiltered output current                   |

And

| Parameter        | Units     | Meaning                                            | Typical value |
|------------------|-----------|----------------------------------------------------|---------------|
| $\tau_\text{RC}$ | ms        | Time constant of filter in $R_s$ compenstation, read from amplifier|0.01 ms|
| $\tau_s$         | ms        | Time constant of stimulus filter, unknown          | 0.025 ms      |
| $\tau_o$         | ms        | Time constant of output filter, see below          | 0.01 ms       |
| $R_f$            | G$\Omega$ | Measurement resistor, from headstage documentation | 0.5 G$\Omega$ |

To estimate $\tau_s$ in a real amplifier, you can record the filtered output voltage and fit to it.
The value above corresponds to the "slow" (default) setting on a HEKA EPC-10.
The value for the "fast" setting is 0.003ms.

To estimate $\tau_o$, you can use $\tau_o \approx \frac{1}{2 \pi f}$, where $f$ is the filter's cut-off frequency in kHz.

Note that exact values of $\tau_s$ and $\tau_o$ are seldom crucial.

The given $R_f$ value is typical for Axon, HEKA, and Sutter.

## Level 0: The full model

Next, we jump straight to the Level 0 model.

<img src="./img/vc-level-0.png" style="margin:auto" />

This level adds

- A pipette (conventional patch clamp) or parasitic (planar patch clamp) capacitance $C_p$, and its amplifier estimate $C^*_p$
- A stray capacitance on the measurement resistor $\tilde{C}_f$, and a finite op-amp speed with time constant $\tau_a$
- A two-part output filter, where "Filter 1" affects series resistance compensation, while "Filter 2" does not.
- Arbitrary equations for the output filters and the stimulus filter (allowing e.g. Bessel formulations)

\begin{align}
0.1. && C_m\dot{V}_m = \frac{V_p + E_\text{off}^\dagger - V_m}{R_s} - I_\text{leak} - I
\end{align}

\begin{align}
0.2. && (C_p+\tilde{C}_f)\dot{V}_p = \frac{V_o - V_p}{R_f} - \frac{V_p + E_\text{off}^\dagger - V_m}{R_s} + \tilde{C}_f\dot{V}_o + C_m^* \dot{V}_e + C_p^* \dot{V}_r
\end{align}

\begin{align}
0.3. && \tau_a \dot{V}_o = V_r - V_p
\end{align}

\begin{align}
0.4. && V_r = V_s + \alpha \frac{R_s^*}{R_f} V_\text{rc} + \beta R_s^* C_m^* \dot{V}_e
              = \alpha \frac{R_s^*}{R_f} V_\text{rc} + \frac{V_s - \beta V_e}{1 - \beta}
\end{align}

\begin{align}
0.5. && \dot{V}_e &= \frac{V_s - V_e}{(1 - \beta)R_s^*C_m^*}   
\end{align}

\begin{align}
0.6. && \tau_\text{rc} \dot{V}_\text{rc} = V_1 - V_\text{rc}
\end{align}

\begin{align}
0.7. && \dot{V_s} = f_s(V_c, V_s)
\end{align}

\begin{align}
0.8. && \dot{V}_1 = f_1(V_o - V_r, V_1)
\end{align}

\begin{align}
0.9. && \dot{V}_2 = f_2(V_1, V_2)
\end{align}

\begin{align}
0.10. && I_\text{obs} = V_2 / R_f
\end{align}

New variables and parameters are

| Variable | Units | Meaning                |
|----------|-------|------------------------|
| $V_o$    | mV    | Op-amp output voltage  |
| $V_p$    | mV    | Pipette potential      |
| $V_1$    | mV    | Voltage after filter 1 |
| $V_2$    | mV    | Voltage after filter 2 |


And

| Parameter     | Units | Meaning                                            | Typical value |
|---------------|-------|----------------------------------------------------|---------------|
| $C_p$         | pF    | True pipette capacitance, unknown                  | 5 pF          |
| $C^*_p$       | pF    | Estimated pipette capacitance, read from amplifier | 5 pF          |
| $\tilde{C}_f$ | pF    | Compensated stray capacitance on measuring resistor, unknown | 5e-3 pF |
| $\tau_a$      | ms    | Time constant of measurement op-amp, unknown       | 13e-6 ms      |

The given values of $\tilde{C}_f$ and $\tau_a$ were estimated by fitting to model cell experiments with a Heka EPC-10.